# promptframe — Example Usage

This notebook walks through every major feature of the library:

1. **PromptRegistry** — load and use YAML prompt files
2. **StructuredPromptBuilder** — compose prompts from components
3. **LLMBaseModel** — generate structured input/output schemas for LLMs
4. **SkillRegistry** — load and inject markdown skill files
5. **Parsers** — parse JSON from LLM responses
6. **CLI** — quick reference for shell commands

> **Setup:** `pip install promptframe` (or install from source)


In [1]:
# Standard imports used throughout the notebook
import json, textwrap, pathlib, tempfile, os

# Create a temporary working directory for sample files
TMP = pathlib.Path(tempfile.mkdtemp())
print('Working directory:', TMP)

Working directory: C:\Users\apmal\AppData\Local\Temp\tmpcmrf7mqe


---
## 1. PromptRegistry — YAML-based Prompt Management

In [10]:
# ── 1a. Create sample YAML prompt files ──────────────────────────────────────

PROMPTS_DIR = "prompts"
COMMON_DIR  = os.path.join(PROMPTS_DIR, "common")
PROD_DIR    = os.path.join(PROMPTS_DIR, "prod")


# # common/summarise.yaml
# (COMMON_DIR / 'summarise.yaml').write_text(textwrap.dedent("""\
#     version: 1.0
#     metadata:
#       type: prompt
#       name: summarise
#       description: Text summarisation prompts
#       tags: [nlp, summarise]
#       project: demo

#     prompts:
#       - pid: short_summary
#         description: Summarise text in one sentence.
#         input_variables: [text]
#         prompt: |
#           Summarise the following text in a single sentence:

#           {text}

#       - pid: bullet_summary
#         description: Summarise as bullet points.
#         input_variables: [text, n]
#         prompt: |
#           Summarise the following text as exactly {n} bullet points:

#           {text}
# """))

# # prod/summarise.yaml — environment override with a tighter prompt
# (PROD_DIR / 'summarise.yaml').write_text(textwrap.dedent("""\
#     version: 1.0
#     metadata:
#       type: prompt
#       name: summarise
#       description: Production summarisation prompts (tighter)
#       tags: [nlp, summarise]
#       project: demo

#     prompts:
#       - pid: short_summary
#         description: Summarise text in one sentence — PROD version.
#         input_variables: [text]
#         prompt: |
#           You are a professional editor. Summarise in one sentence, no jargon:

#           {text}

#       - pid: bullet_summary
#         description: Summarise as bullet points — PROD version.
#         input_variables: [text, n]
#         prompt: |
#           You are a professional editor. Produce exactly {n} concise bullet points:

#           {text}
# """))

# print('Prompt files created.')

In [11]:
# ── 1b. Load prompts via PromptRegistry ──────────────────────────────────────
from promptframe import PromptRegistry

# common environment (no override)
reg_common = PromptRegistry(base=str(PROMPTS_DIR), common='common')
p_common = reg_common.load_prompt('summarise')
print('Common version:', repr(p_common))

# prod environment (overrides common)
reg_prod = PromptRegistry(base=str(PROMPTS_DIR), environment='prod', common='common')
p_prod = reg_prod.load_prompt('summarise')
print('Prod version:  ', repr(p_prod))

Common version: PromptYAML(name='summarise', prompts=['short_summary', 'bullet_summary'])
Prod version:   PromptYAML(name='summarise', prompts=['short_summary', 'bullet_summary'])


In [12]:
# ── 1c. Attribute access and formatting ──────────────────────────────────────

# Access a prompt by pid as an attribute
prompt = p_prod.short_summary
print('Prompt text:\n', prompt.prompt)

# Format with variables
filled = prompt.format(text="The quick brown fox jumps over the lazy dog.")
print('\nFormatted:\n', filled)

# Bullet summary
bullet = p_prod.bullet_summary.format(
    text="Climate change is accelerating. Polar ice is melting. Sea levels are rising.",
    n=3
)
print('\nBullet summary prompt:\n', bullet)

Prompt text:
 Summarise the following text in a single sentence:

{text}


Formatted:
 Summarise the following text in a single sentence:

The quick brown fox jumps over the lazy dog.


Bullet summary prompt:
 Summarise the following text as exactly 3 bullet points:

Climate change is accelerating. Polar ice is melting. Sea levels are rising.



In [13]:
# ── 1d. List available prompt files ──────────────────────────────────────────
print('Files found by registry:', reg_prod.list_prompts())

Files found by registry: ['summarise.yaml']


---
## 2. StructuredPromptBuilder — Composing Prompts

In [14]:
from promptframe.builder import StructuredPromptBuilder
from promptframe.components.basic import (
    SimplePromptComponent,
    PromptSectionComponent,
    InputComponent,
    ConditionalPromptComponent,
    TemplatePromptComponent,
)

In [15]:
# ── 2a. SimplePromptComponent ────────────────────────────────────────────────

builder = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are a helpful assistant.")
    >> SimplePromptComponent("Answer the following question: {question}")
)

result = builder.build({"question": "What is the capital of France?"})
print(result)

You are a helpful assistant.

Answer the following question: What is the capital of France?


In [16]:
# ── 2b. PromptSectionComponent — bullet rules ─────────────────────────────────

rules = PromptSectionComponent(
    requirement=["Be concise.", "Avoid jargon.", "Cite sources."],
    header="Rules:",
)

builder2 = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are a research assistant.")
    >> rules
    >> InputComponent(header="Your input:", template="<query>{query}</query>")
)

print(builder2.build({"query": "Explain quantum entanglement."}))

You are a research assistant.

Rules:
- Be concise.
- Avoid jargon.
- Cite sources.

Your input:
<query>Explain quantum entanglement.</query>


In [17]:
# ── 2c. ConditionalPromptComponent ───────────────────────────────────────────

json_instruction = ConditionalPromptComponent(
    component=SimplePromptComponent("Respond ONLY with a valid JSON object."),
    condition_key="json_mode",
)

builder3 = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are an extraction assistant.")
    >> json_instruction
    >> InputComponent()
)

# json_mode ON
print("--- json_mode=True ---")
print(builder3.build({"json_mode": True, "input": "Alice, 30, engineer"}))

# json_mode OFF
print("\n--- json_mode=False ---")
print(builder3.build({"json_mode": False, "input": "Alice, 30, engineer"}))

--- json_mode=True ---
You are an extraction assistant.

Respond ONLY with a valid JSON object.

Input for processing is given below.
<input>Alice, 30, engineer</input>

--- json_mode=False ---
You are an extraction assistant.

Input for processing is given below.
<input>Alice, 30, engineer</input>


In [18]:
# ── 2d. TemplatePromptComponent — compose named slots ────────────────────────

tmpl = TemplatePromptComponent(
    template="System: {system}\n\nTask: {task}",
    components={
        "system": SimplePromptComponent("You are a senior Python developer."),
        "task":   SimplePromptComponent("Write a function that {action}."),
    },
)

print(tmpl.render({"action": "sorts a list of dicts by a given key"}))

System: You are a senior Python developer.

Task: Write a function that sorts a list of dicts by a given key.


In [19]:
# ── 2e. preview() — inspect each component separately ───────────────────────

builder2.preview({"query": "Explain quantum entanglement."})

🔍 Prompt Preview
[0] SimplePromptComponent
------------------------------
You are a research assistant.

[1] PromptSectionComponent
------------------------------
Rules:
- Be concise.
- Avoid jargon.
- Cite sources.

[2] InputComponent
------------------------------
Your input:
<query>Explain quantum entanglement.</query>



---
## 3. LLMBaseModel — Structured Input/Output Schemas

In [20]:
from typing import Optional, List
from promptframe.llm_base_model import LLMBaseModel
from promptframe.fields import LLMField


class Address(LLMBaseModel):
    street: str = LLMField(..., description="Street address line.")
    city:   str = LLMField(..., description="City name.")
    zip:    str = LLMField(..., description="Postal / ZIP code.")


class Customer(LLMBaseModel):
    name:    str            = LLMField(
        ...,
        description="Full customer name.",
        model_attribute_id="customer_name",
        output_instruction="Return a normalised name in Title Case.",
    )
    email:   str            = LLMField(..., description="Customer email address.")
    age:     Optional[int]  = LLMField(None, description="Age in years.")
    address: Address        = LLMField(..., description="Customer shipping address.")
    tags:    List[str]      = LLMField(default_factory=list, description="Descriptive tags.")


print('Model defined: Customer')

Model defined: Customer


In [21]:
# ── 3a. Input instructions (what to send TO the model) ──────────────────────

print(Customer.get_input_instructions())


Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "Full customer name."
  },
  "email": {
    "instruction": "Customer email address."
  },
  "age": {
    "instruction": "Age in years."
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Street address line."
      },
      "city": {
        "instruction": "City name."
      },
      "zip": {
        "instruction": "Postal / ZIP code."
      }
    }
  },
  "tags": {
    "instruction": "Descriptive tags."
  }
}</input_schema>



In [22]:
# ── 3b. Output format instructions (how the model should reply) ──────────────

print(Customer.get_format_instructions())

Your response must be a valid JSON parseable object.
This ensures the output can be reliably parsed and used in downstream processes.

Example of a JSON Schema is shown below:
{
  "properties": {
    "user": {
      "type": "object",
      "properties": {
        "id": {"type": "integer"},
        "profile": {
          "type": "object",
          "properties": {
            "name": {"type": "string"},
            "skills": {"type": "array", "items": {"type": "string"}}
          },
          "required": ["name", "skills"]
        }
      },
      "required": ["id", "profile"]
    }
  },
  "required": ["user"]
}

Valid output:
{
  "user": {
    "id": 123,
    "profile": {
      "name": "Alice",
      "skills": ["Python", "FastAPI"]
    }
  }
}

Your response should be STRICLY formated using this schema:

<format_instructions>"{\n  \"$defs\": {\n    \"Address\": {\n      \"properties\": {\n        \"street\": {\n          \"title\": \"Street\",\n          \"type\": \"string\",\n        

In [23]:
# ── 3c. get_llm_schema — both in one dict ────────────────────────────────────

schema = Customer.get_llm_schema(get_dict=True)
print(json.dumps(schema, indent=2))

{
  "input": {
    "name": {
      "instruction": "Full customer name."
    },
    "email": {
      "instruction": "Customer email address."
    },
    "age": {
      "instruction": "Age in years."
    },
    "address": {
      "instruction": "Customer shipping address.",
      "fields": {
        "street": {
          "instruction": "Street address line."
        },
        "city": {
          "instruction": "City name."
        },
        "zip": {
          "instruction": "Postal / ZIP code."
        }
      }
    },
    "tags": {
      "instruction": "Descriptive tags."
    }
  },
  "output": {
    "$defs": {
      "Address": {
        "properties": {
          "street": {
            "title": "Street",
            "type": "string",
            "output_instruction": "Street address line."
          },
          "city": {
            "title": "City",
            "type": "string",
            "output_instruction": "City name."
          },
          "zip": {
            "title": "Zip"

In [24]:
# ── 3d. Ignoring fields ───────────────────────────────────────────────────────

# Exclude 'age' and nested 'address.zip' from instructions
partial = Customer.get_input_instructions(get_dict=True, ignore=('age', 'address.zip'))
print(json.dumps(partial, indent=2))

{
  "name": {
    "instruction": "Full customer name."
  },
  "email": {
    "instruction": "Customer email address."
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Street address line."
      },
      "city": {
        "instruction": "City name."
      }
    }
  },
  "tags": {
    "instruction": "Descriptive tags."
  }
}


In [25]:
# ── 3e. Binding model_prompt YAML to LLMBaseModel fields ────────────────────

# Create a model_prompt YAML file
# (TMP / 'customer_prompts.yaml').write_text(textwrap.dedent("""\
#     version: 1.0
#     metadata:
#       type: model_prompt
#       name: customer_prompts
#       description: Field-level prompts for the Customer model.
#       tags: [customer]
#       project: demo

#     prompts:
#       - pid: name_prompt
#         model_attribute_id: customer_name
#         input_instruction: |
#           The raw input may contain a messy or abbreviated name.
#         output_instruction: |
#           Return the customer's full name in Title Case, e.g. 'John Smith'.
# """))
reg = PromptRegistry(base=str(PROMPTS_DIR), common='common')
mp  = reg.load_model_prompt('customer_prompts')

# Now generate instructions with prompt-level overrides
print(Customer.get_input_instructions_with_prompt(
    get_dict=False,
    prompt_model_dict=mp.prompt_model_dict,
))


Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "The raw input may contain a messy or abbreviated name.\n"
  },
  "email": {
    "instruction": "Customer email address."
  },
  "age": {
    "instruction": "Age in years."
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Street address line."
      },
      "city": {
        "instruction": "City name."
      },
      "zip": {
        "instruction": "Postal / ZIP code."
      }
    }
  },
  "tags": {
    "instruction": "Descriptive tags."
  }
}</input_schema>



---
## 4. SkillRegistry — Markdown Skill Files

In [ ]:
# ── 4a. Create sample skill files ────────────────────────────────────────────

SKILLS_DIR = TMP / 'skills'

# Folder-based skill
(SKILLS_DIR / 'python-expert').mkdir(parents=True)
(SKILLS_DIR / 'python-expert' / 'SKILL.md').write_text(textwrap.dedent("""\
    ---
    name: Python Expert
    description: Best practices for writing production-quality Python code.
    tags:
      - python
      - backend
    version: "1.0"
    ---

    ## Code Style

    Always follow PEP 8. Use type hints for all public functions.
    Prefer `pathlib.Path` over `os.path`.

    ## Error Handling

    Raise specific exceptions. Never use bare `except:` clauses.
    Log errors with context using the `logging` module.

    ## Testing

    Write tests with `pytest`. Aim for 80%+ coverage on business logic.
"""))

# Flat-file skill
SKILLS_DIR.mkdir(exist_ok=True)
(SKILLS_DIR / 'data-analysis.md').write_text(textwrap.dedent("""\
    ---
    name: Data Analysis
    description: Guidelines for exploratory data analysis tasks.
    tags:
      - data
      - pandas
    version: "1.0"
    ---

    ## Data Loading

    Always inspect shape, dtypes, and null counts before analysis.

    ## Visualisation

    Use matplotlib or seaborn. Label all axes. Include a title.
"""))

print('Skill files created.')

In [ ]:
# ── 4b. List and load skills ──────────────────────────────────────────────────

from promptframe import SkillRegistry

skill_reg = SkillRegistry(str(SKILLS_DIR))

print('Discovered skills:')
for entry in skill_reg.list():
    print(f"  {entry['key']:20s} — {entry['description']}")

In [ ]:
# ── 4c. Load a skill and inspect sections ────────────────────────────────────

skill = skill_reg.get('python-expert')
print(repr(skill))
print('\nSections:', list(skill.sections.keys()))
print('\nError Handling section:\n', skill.get_section('Error Handling'))

In [ ]:
# ── 4d. Inject a skill into a prompt via SkillComponent ──────────────────────

from promptframe.components.basic import SkillComponent

skill = skill_reg.get('python-expert')

prompt = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are a senior Python engineer.")
    >> SkillComponent(
           skill,
           sections=["Code Style", "Error Handling"],
           wrapper="<guidelines>\n{skill}\n</guidelines>",
       )
    >> InputComponent(header="Task:", template="{task}")
).build({"task": "Write a CSV reader that handles missing values."})

print(prompt)

In [ ]:
# ── 4e. Load skill via PromptRegistry.load_skill ─────────────────────────────

reg2 = PromptRegistry(base=str(TMP))
da_skill = reg2.load_skill('skills/data-analysis.md')
print(da_skill.render(include_name=True))

---
## 5. Parsers — JSON from LLM Responses

In [ ]:
from promptframe.parsers import json_parser, parse_partial_json, parse_json_markdown

# ── 5a. Standard JSON response ────────────────────────────────────────────────
raw = '{"name": "Alice", "age": 30}'
print(json_parser(raw))

In [ ]:
# ── 5b. Markdown-fenced JSON (common LLM output) ─────────────────────────────
fenced = """
Sure! Here is your answer:

```json
{"city": "Paris", "country": "France"}
```
"""
print(parse_json_markdown(fenced))

In [ ]:
# ── 5c. Partial / truncated JSON ─────────────────────────────────────────────
truncated = '{"name": "Bob", "scores": [10, 20'
print(parse_partial_json(truncated))

---
## 6. Full End-to-End Example

Build a complete extraction prompt for a `Customer` model using all features together.

In [ ]:
# Reload everything cleanly
reg       = PromptRegistry(base=str(TMP))
skill_reg = SkillRegistry(str(SKILLS_DIR))
py_skill  = skill_reg.get('python-expert')
mp        = reg.load_model_prompt('customer_prompts')

# Input schema with prompt overrides
input_schema  = Customer.get_input_instructions_with_prompt(
    prompt_model_dict=mp.prompt_model_dict
)
# Output schema
output_schema = Customer.get_format_instructions_with_prompt(
    prompt_model_dict=mp.prompt_model_dict
)

final_prompt = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are a data extraction assistant.")
    >> SimplePromptComponent(input_schema)
    >> SimplePromptComponent(output_schema)
    >> InputComponent(
           header="Extract structured data from the text below:",
           template="<text>{raw_text}</text>"
       )
).build({
    "raw_text": "alice smith, alice@example.com, 28, 123 Main St, Springfield, 62701"
})

print(final_prompt)

In [ ]:
# Simulate parsing an LLM response
mock_llm_response = """
```json
{
  "name": "Alice Smith",
  "email": "alice@example.com",
  "age": 28,
  "address": {"street": "123 Main St", "city": "Springfield", "zip": "62701"},
  "tags": ["new-customer"]
}
```
"""

parsed = json_parser(mock_llm_response)
customer = Customer(**parsed)
print(customer)
print('Name:', customer.name)
print('City:', customer.address.city)

---
## 7. CLI Quick Reference

```bash
# Scaffold a prompt directory tree
promptframe scaffold prompts/ --example

# Initialise a new prompt file
promptframe init regular prompts/common/my_prompts.yaml

# List all prompt files in a directory
promptframe list prompts/

# Validate all YAML files
promptframe validate prompts/

# Inspect a specific file
promptframe inspect prompts/common/my_prompts.yaml

# Render a specific prompt
promptframe render prompts/common/my_prompts.yaml short_summary

# Lint (check for missing descriptions)
promptframe lint prompts/

# Export to JSON
promptframe export prompts/common/my_prompts.yaml --format json -o out.json

# Diff two prompt files
promptframe diff prompts/common/v1.yaml prompts/common/v2.yaml

# Skill commands
promptframe skill init python-expert --path skills/
promptframe skill list skills/
promptframe skill inspect python-expert --path skills/
promptframe skill render python-expert --path skills/ --section "Code Style"
promptframe skill validate skills/
promptframe skill lint skills/
promptframe skill search python --path skills/
promptframe skill diff skills/v1/SKILL.md skills/v2/SKILL.md
```